# PPLM CEFR Latent Classifier Training

This notebook trains the lightweight CEFR attribute classifier used by the PPLM baseline.

The classifier operates on 4096-dimensional hidden-state representations extracted from the frozen `Llama-3.1-8B-Instruct` backbone. These latent representations are loaded from the PPLM latent dataset constructed during data preparation.

The classifier consists of a single linear layer preceded by dropout and predicts one of the six CEFR proficiency levels: A1, A2, B1, B2, C1, and C2.

Training uses a custom **Ordinal Weighted Loss** combining:

- weighted cross-entropy to compensate for CEFR class imbalance; and
- an ordinal Mean Squared Error penalty based on the expected CEFR class value.

The dataset is partitioned into an 85% training split and a 15% validation split using stratification and `random_state=42`.

This notebook is responsible only for training and saving the PPLM latent classifier. Detailed evaluation of the trained classifier is performed separately.

In [ ]:
!pip install -q torch datasets huggingface_hub scikit-learn tqdm

import os
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from huggingface_hub import login, HfApi
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from tqdm import tqdm


In [ ]:
# =========================================================
# 1. AUTHENTICATION & ENVIRONMENT CONFIGURATION
# =========================================================
# change to actual hf token
hf_token = "HF_Token"
login(token=hf_token)

# 🌟 RENAME TO 1-LAYER REPOSITORY NAME 🌟
model_repo_id = "MohammadKhosravi/llama3.1-8b-cefr-steering-1layer-head-ordinal-universal"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using execution device: {device}")

Using execution device: cuda


In [ ]:
# =========================================================
# 2. DATA MANIFOLD INGESTION & PARSING
# =========================================================
print("\nDownloading unified vector dataset from Hugging Face...")
dataset = load_dataset("MohammadKhosravi/cefr-llama3.1-8b-hidden-states-efcamdat-universal", split="train")

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
reverse_map = {0: "A1", 1: "A2", 2: "B1", 3: "B2", 4: "C1", 5: "C2"}

print("Parsing vector matrices and alignment maps...")
X_list = []
y_list = []

for row in dataset:
    X_list.append(row["hidden_vector"])
    raw_label = row["cefr_level"]
    clean_label = raw_label[:2]  # Truncates any lingering string artifacts cleanly down to A1-C2
    y_list.append(label_map[clean_label])

# Construct dense tensors
X = torch.tensor(X_list, dtype=torch.float32)
y = torch.tensor(y_list, dtype=torch.long)

# Protect calculation path against dynamic memory corruption artifacts
X = torch.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

README.md:   0%|          | 0.00/336 [00:00<?, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/242M [00:00<?, ?B/s]

data/train-00004-of-00005.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/66494 [00:00<?, ? examples/s]

Parsing vector matrices and alignment maps...


In [ ]:
# =========================================================
# 3. COMPILING SPLITS & COMPUTING CLASS WEIGHTS
# =========================================================
print("Partitioning data into Train (85%) and Validation (15%) splits...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

print("Computing inverse-frequency balancing class weights dynamically...")
y_train_np = y_train.numpy()
classes = np.unique(y_train_np)

# Exact class weight allocation for your specific dataset distribution
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_np)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

print(f" -> Dynamically Computed Class Weights: {weights}")

# Build loader utilities
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

Partitioning data into Train (85%) and Validation (15%) splits...
Computing inverse-frequency balancing class weights dynamically...
 -> Dynamically Computed Class Weights: [0.73863666 0.73719153 0.73408926 0.73667266 2.21070954 8.50165463]


In [ ]:
# =========================================================
# 4. ARCHITECTURE CONFIGURATION (🌟 1-LAYER LINEAR HEAD 🌟)
# =========================================================
class CEFR1LayerLinearHead(nn.Module):
    def __init__(self, input_dim=4096, num_classes=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Dropout(p=0.35),       # Keeps the exact input dropout to prevent over-indexing
            nn.Linear(input_dim, num_classes) # Straight projection: clean, linear, stable gradients
        )
    def forward(self, x):
        return self.network(x)

model = CEFR1LayerLinearHead().to(device)

In [ ]:
# =========================================================
# 5. CUSTOM ORDINAL WEIGHTED LOSS FUNCTION
# =========================================================
class OrdinalWeightedLoss(nn.Module):
    def __init__(self, weights, lambda_mse=1.0, num_classes=6):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss(weight=weights)
        self.mse_loss = nn.MSELoss()
        self.lambda_mse = lambda_mse
        self.num_classes = num_classes

    def forward(self, logits, targets):
        # 1. Categorical Loss
        ce = self.ce_loss(logits, targets)

        # 2. Ordinal Distance Loss
        probs = torch.softmax(logits, dim=1)
        class_indices = torch.arange(self.num_classes, device=logits.device, dtype=torch.float32)
        expected_class = torch.sum(probs * class_indices, dim=1)

        # Penalize distance mistakes exponentially using MSE
        mse = self.mse_loss(expected_class, targets.float())

        return ce + (self.lambda_mse * mse)

# Loss configuration parameters
criterion = OrdinalWeightedLoss(weights=class_weights_tensor, lambda_mse=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [ ]:
# =========================================================
# 6. CRUNCHING OPTIMIZATION LOOP
# =========================================================
epochs = 50
print(f"\nInitiating Steering Head Optimization (Volume: {len(X_train):,} training vectors)...")

best_val_acc = 0.0
best_val_adj = 0.0
checkpoint_name = "cefr_steering_head_universal_ordinal.pt"

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

    train_acc = 100 * correct / total

    # Validation Engine
    model.eval()
    val_correct_strict = 0
    val_correct_adjacent = 0
    val_total = 0

    with torch.no_grad():
        for val_X, val_y in val_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            val_outputs = model(val_X)
            _, val_predicted = torch.max(val_outputs.data, 1)

            val_total += val_y.size(0)
            val_correct_strict += (val_predicted == val_y).sum().item()

            # Calculate adjacent alignment accuracy boundaries (+/- 1 scale steps)
            distance = torch.abs(val_predicted - val_y)
            val_correct_adjacent += (distance <= 1).sum().item()

    val_acc_strict = 100 * val_correct_strict / val_total
    val_acc_adj = 100 * val_correct_adjacent / val_total

    # Step the learning rate down if strict accuracy plateaus
    scheduler.step(val_acc_strict)

    # Save tracking state weights on performance peaks
    if val_acc_strict > best_val_acc:
        best_val_acc = val_acc_strict
        best_val_adj = val_acc_adj
        torch.save(model.state_dict(), checkpoint_name)

    if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == (epochs - 1):
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1:02d}/{epochs}] | LR: {current_lr:.6f} | Loss: {total_loss/len(train_loader):.4f} | Strict Acc: {val_acc_strict:.2f}% | Adjacent Acc: {val_acc_adj:.2f}%")

print(f"\n🎉 Optimization Complete! Maximum Baseline Strict Validation Accuracy achieved: {best_val_acc:.2f}%")



Initiating Steering Head Optimization (Volume: 56,519 training vectors)...
Epoch [01/50] | LR: 0.000300 | Loss: 1.3687 | Strict Acc: 77.19% | Adjacent Acc: 94.93%
Epoch [05/50] | LR: 0.000300 | Loss: 0.7541 | Strict Acc: 82.68% | Adjacent Acc: 96.25%
Epoch [10/50] | LR: 0.000300 | Loss: 0.6760 | Strict Acc: 83.79% | Adjacent Acc: 96.87%
Epoch [15/50] | LR: 0.000150 | Loss: 0.6453 | Strict Acc: 84.37% | Adjacent Acc: 97.03%
Epoch [20/50] | LR: 0.000150 | Loss: 0.6360 | Strict Acc: 84.21% | Adjacent Acc: 96.83%
Epoch [25/50] | LR: 0.000150 | Loss: 0.6311 | Strict Acc: 85.08% | Adjacent Acc: 97.24%
Epoch [30/50] | LR: 0.000075 | Loss: 0.6125 | Strict Acc: 85.05% | Adjacent Acc: 97.36%
Epoch [35/50] | LR: 0.000019 | Loss: 0.6150 | Strict Acc: 84.94% | Adjacent Acc: 97.18%
Epoch [40/50] | LR: 0.000009 | Loss: 0.6064 | Strict Acc: 85.01% | Adjacent Acc: 97.18%
Epoch [45/50] | LR: 0.000002 | Loss: 0.6143 | Strict Acc: 85.03% | Adjacent Acc: 97.21%
Epoch [50/50] | LR: 0.000001 | Loss: 0.6149 

In [ ]:
# =========================================================
# 7. EXPORT TO HUGGING FACE ARCHIVE (BATCH COMMIT WORKAROUND)
# =========================================================
from huggingface_hub import CommitOperationAdd

# 1. Load state dict back to ensure serialization of peak weights
model.load_state_dict(torch.load(checkpoint_name))
file_name = "cefr_steering_head.pt"
torch.save(model.state_dict(), file_name)

# 2. Automated README Compiling
readme_content = f"""---
language: en
tags:
- feature-extraction
- cefr
- llama-3.1
- steering-head
- ordinal-classification
- controllable-text-generation
- pplm
---

# Llama-3.1-8B CEFR 1-Layer MLP Steering Head (Ordinal Universal)

This repository contains a **1-Layer Linear Classifier Head** designed to steer the hidden representations of `meta-llama/Llama-3.1-8B-Instruct` towards specific CEFR proficiency levels (A1-C2).

## 🧠 Architectural Rationale
Unlike deeper multi-layer MLPs, this **1-layer head** maps hidden representations ($d=4096$) directly to CEFR logits ($d=6$) using a single linear layer with an input dropout of 0.35. By eliminating ReLU activations and LayerNorm scaling, this architecture ensures:
1. **Strict Gradient Continuity:** Eliminates vanishing/exploding gradient boundaries.
2. **Stable Backpropagation Landscape:** The continuous linear gradient vector provides stable, non-saturating paths for real-time decoding-time token perturbation (PPLM).

## 📊 Training Specifications & Performance
* **Loss Function:** Custom Ordinal Weighted Loss (Weighted Cross-Entropy + MSE Ordinal Distance Penalty)
* **Optimizer:** AdamW (Learning Rate: 3e-4, Weight Decay: 0.01)
* **Epochs:** 50
* **Batch Size:** 128
* **Peak Validation Strict Accuracy:** **{best_val_acc:.2f}%**
* **Peak Validation Adjacent Accuracy:** **{best_val_adj:.2f}%**
"""
readme_file = "README.md"
with open(readme_file, "w") as f:
    f.write(readme_content)

# 3. Batch Upload via Single Commit
print(f"\nPushing bundled files to Hugging Face Hub: {model_repo_id}...")
api = HfApi()
api.create_repo(repo_id=model_repo_id, exist_ok=True, private=False)

operations = [
    CommitOperationAdd(path_in_repo=file_name, path_or_fileobj=file_name),
    CommitOperationAdd(path_in_repo=readme_file, path_or_fileobj=readme_file)
]

try:
    api.create_commit(
        repo_id=model_repo_id,
        operations=operations,
        commit_message="Initial commit: 1-layer ordinal steering head and README"
    )
    print(f"✅ Deployment complete! Track it here: https://huggingface.co/{model_repo_id}")
except Exception as e:
    print(f"❌ API still failing. Please use Option 2 (Manual Upload). Error: {e}")


Pushing bundled files to Hugging Face Hub: MohammadKhosravi/llama3.1-8b-cefr-steering-1layer-head-ordinal-universal...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ent/cefr_steering_head.pt: 100%|##########|  100kB /  100kB            

✅ Deployment complete! Track it here: https://huggingface.co/MohammadKhosravi/llama3.1-8b-cefr-steering-1layer-head-ordinal-universal
